# AnomalyMatch on real Chandra Source Catalog cutouts (GPU)

**Why this notebook exists:** this is Setup Step 2 of the `image_anomaly_detection` project in `chandra-toolkit` - after validating the AnomalyMatch pipeline mechanics on GalaxyMNIST (`anomalymatch_galaxymnist_kaggle.ipynb`, Setup Step 1), this notebook runs the same pipeline on **real Chandra X-ray Observatory image cutouts** for the first time - a genuinely novel data product for this method (AnomalyMatch has previously been applied to Hubble, JWST, and Euclid, but not X-ray/Chandra data).

**Data and label caveat, read before interpreting results:** there is no existing human-vetted "anomaly" label for Chandra sources. As a proxy, this notebook uses the CSC 2.1 catalog's own `extent_flag` (extended vs. point-like source, a real measured property from the catalog pipeline) as the two classes - extended sources are the "anomaly" class, point-like sources are "normal". This is a real, meaningful morphology distinction, but it is a proxy, not a vetted anomaly ground truth - treat results here as a pipeline/method validation on real data, not a discovery claim.

**Known imaging artifact, also read before interpreting results:** CSC's exposure-corrected image product (`ecorrimg`) carries real ACIS "frame-transfer streak" instrumental artifacts (a documented Chandra CCD readout effect) - visible as faint diagonal lines and localized sharp value spikes near bright sources. This was diagnosed in depth while building the cutout pipeline (see `image_anomaly_detection/PLAN.md`, "Setup step 2 status") and is not fixable without full CIAO reprocessing software. It's accepted here as representative real-world imaging noise (it appears independent of the extent_flag label, so shouldn't create a spurious shortcut) rather than blocking on a multi-GB software install.

**Before running:** in the Kaggle notebook settings panel (right sidebar), set **Accelerator -> GPU T4 x2**. Do NOT use P100 - current PyTorch wheels have dropped compiled kernels for Pascal (`sm_60`), which caused a `CUDA error: no kernel image is available for execution on the device` failure when this was tried (see the GalaxyMNIST notebook's own history for that debugging chain).

## What method is being tested: AnomalyMatch

[AnomalyMatch](https://github.com/esa/AnomalyMatch) (Gomez et al., [arXiv:2505.03509](https://arxiv.org/abs/2505.03509), ESA) is a semi-supervised anomaly-detection method for astronomical images, built from three pieces:

1. **Backbone model**: an EfficientNet image classifier (via the `timm` library), trained as a binary "normal vs. anomaly" classifier.
2. **FixMatch** (Sohn et al. 2020): a semi-supervised training algorithm that exploits a large *unlabeled* pool alongside a small labeled set, via consistency regularization + pseudo-labeling on weakly/strongly augmented image pairs.
3. **Active learning loop**: after each training run, the model scores every unlabeled image, top-scoring ambiguous examples are surfaced and corrected, and the model retrains - repeated for several "training runs".

Already applied by its original authors to Hubble (99.6M cutouts), JWST (lens candidates), and Euclid Q1 (jellyfish galaxies) - **not yet to any Chandra X-ray data**, which is what this notebook tests for the first time.

## What data is being used: Chandra Source Catalog 2.1 image cutouts

Real CSC 2.1 sources, queried live via `pyvo` against Chandra's public TAP/SIA services (`cda.cfa.harvard.edu`):

- **Point-like sources** (`extent_flag=0`): the "normal" class.
- **Extended sources** (`extent_flag=1`): the "anomaly" class - real, measured extended X-ray morphology (e.g. diffuse emission, superimposed sources, extended remnants), not synthetic.
- Both filtered to `conf_flag=0 AND significance>10` to avoid marginal/confused detections.

Each source's image comes from CSC's SIA endpoint (`csc21siap/queryImages`) as a full CCD-frame exposure-corrected image, WCS-cropped to a ~90 arcsec cutout centered on the source, median-filtered (suppresses the streak artifact described above) and asinh-stretched to an 8-bit RGB JPEG. All of this logic lives in `chandra-toolkit`'s `common/data_access.py` and `image_anomaly_detection/build_seed_cutouts.py` - cloned and reused directly below rather than reimplemented in-notebook, so it's the exact same tested code path, not a notebook-only copy.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU detected - set Accelerator to GPU T4 x2 in the notebook settings "
    "panel and restart the session before continuing. Do NOT use P100 (see markdown above)."
)

## 1. Clone chandra-toolkit and AnomalyMatch, apply known bug fixes

`chandra-toolkit` provides the real Chandra data-access/cutout-building code (`common/`, `image_anomaly_detection/build_seed_cutouts.py`). `AnomalyMatch` provides the FixMatch+active-learning benchmark harness. The patches below are the same real, hardware-independent bugs found while validating the GalaxyMNIST pipeline (`.safetensors` vs `.pth`, TurboJPEG's native lib missing), plus two new patches specific to running a dataset AnomalyMatch's benchmark script doesn't know about out of the box: its `--dataset` argument and `load_dataset_info()` only recognize `galaxymnist`/`galaxyzoo`/`miniimagenet` by name - we add a `chandra` option pointing at the files this notebook builds in step 2.

In [ ]:
import os
from pathlib import Path

if not Path("chandra-ml-toolkit").exists():
    !git clone --depth 1 https://github.com/rajul-kk/chandra-ml-toolkit.git
if not Path("AnomalyMatch").exists():
    !git clone --depth 1 https://github.com/esa/AnomalyMatch.git
%cd AnomalyMatch
!pip install -q . pyvo astropy scipy

In [ ]:
from pathlib import Path

pb_path = Path("paper_scripts/paper_benchmark.py")
pu_path = Path("paper_scripts/paper_utils.py")

# --- paper_benchmark.py: same known fixes as the GalaxyMNIST notebook ---
src = pb_path.read_text()

if "faulthandler.enable()" not in src:
    src = src.replace('"""\n\nimport os\n',
                       '"""\n\nimport faulthandler\nfaulthandler.enable()\n\nimport os\n', 1)

src = src.replace(
    'jpeg_decoder = TurboJPEG()\n'
    'USE_TURBOJPEG = True\n'
    'logger.info("Using TurboJPEG for faster image decoding")',
    'try:\n'
    '    jpeg_decoder = TurboJPEG()\n'
    '    USE_TURBOJPEG = True\n'
    '    logger.info("Using TurboJPEG for faster image decoding")\n'
    'except Exception as e:\n'
    '    jpeg_decoder = None\n'
    '    USE_TURBOJPEG = False\n'
    '    logger.warning(f"TurboJPEG unavailable ({e}); falling back to PIL")'
)

src = src.replace('f"model_iter{iteration + 1}.pth"', 'f"model_iter{iteration + 1}.safetensors"')
src = src.replace('f"model.pth"', 'f"model.safetensors"')
src = src.replace('f"model_iteration_{iteration}.pth"', 'f"model_iteration_{iteration}.safetensors"')

src = src.replace(
    'batch_size = 1000  # Lowered batch size to avoid 32-bit indexing error from PyTorch',
    'batch_size = int(os.environ.get("ANOMALYMATCH_PRED_BATCH_SIZE", "1000"))',
)
pb_path.write_text(src)

# --- paper_utils.py: known fixes + new 'chandra' dataset support ---
src = pu_path.read_text()
if "import torch\n" not in src:
    src = src.replace("import pandas as pd\n", "import pandas as pd\nimport torch\n", 1)
src = src.replace(
    "cfg.num_workers = 4\n    cfg.pin_memory = True",
    'cfg.num_workers = int(os.environ.get("ANOMALYMATCH_NUM_WORKERS", "4"))\n'
    "    cfg.pin_memory = torch.cuda.is_available()",
)

src = src.replace(
    'choices=["galaxymnist", "miniimagenet", "galaxyzoo"],',
    'choices=["galaxymnist", "miniimagenet", "galaxyzoo", "chandra"],',
)
src = src.replace(
    'if args.dataset == "galaxymnist":',
    'if args.dataset == "chandra":\n'
    '        labels_path = os.path.join(base_path, "chandra", "labels_chandra.csv")\n'
    '        data_dir = os.path.join(base_path, "chandra")\n'
    '        hdf5_path = os.path.join(base_path, "chandra", "chandra_224.hdf5")\n'
    '    elif args.dataset == "galaxymnist":',
)
pu_path.write_text(src)

assert "chandra" in pu_path.read_text()
print("Patches applied.")

## 2. Build the real Chandra cutout pool

Runs the exact same `build_seed_cutouts.py` used to validate this pipeline locally (query CSC by `extent_flag` -> SIA download -> WCS crop -> normalize), then packs the resulting JPEGs into the HDF5 + labels-CSV format `paper_benchmark.py` expects, via AnomalyMatch's own `create_hdf5_file` helper (same format GalaxyMNIST/miniImageNet prep uses).

`n_extended`/`n_point` below are deliberately larger than a quick sanity check (real dedup by shared observation field keeps bandwidth reasonable - about 185 unique ~50-90MB field downloads produced 474 cutouts when this was last run locally) but smaller than GalaxyMNIST's 10,000 - CSC doesn't have millions of these to draw from at this significance/quality threshold, and each is a real network download, not a bundled dataset.

In [ ]:
import sys
sys.path.insert(0, "../chandra-ml-toolkit")
from image_anomaly_detection.build_seed_cutouts import build as build_chandra_pool

chandra_pool_dir = Path("chandra_pool")
build_chandra_pool(n_extended=40, n_point=460, out_dir=chandra_pool_dir)

In [ ]:
import shutil
import pandas as pd
from PIL import Image
sys.path.insert(0, "paper_scripts")
from prepare_datasets import create_hdf5_file

labels_df = pd.read_csv(chandra_pool_dir / "labels_chandra.csv")

out_dir = Path("paper_scripts/datasets/chandra")
out_dir.mkdir(parents=True, exist_ok=True)
labels_df.to_csv(out_dir / "labels_chandra.csv", index=False)

# anomaly_match's own dataset loader scans cfg.data_dir as a plain folder of
# loose image files (AnomalyDetectionDataset.get_image_names_from_folder) -
# it does NOT read the HDF5 for that. GalaxyMNIST's own prep script writes
# both a loose-file folder (save_images_to_folder) and the HDF5
# (create_hdf5_file); build_seed_cutouts.py only produced the latter, so
# copy the already-built JPEGs into data_dir too, or the loader finds 0
# images despite the HDF5 being correctly populated.
for fn in labels_df["filename"]:
    shutil.copy(chandra_pool_dir / "images" / fn, out_dir / fn)

images = [Image.open(chandra_pool_dir / "images" / fn) for fn in labels_df["filename"]]
create_hdf5_file(images, labels_df["filename"].tolist(), out_dir / "chandra_224.hdf5", img_size=224)

print(f"{len(labels_df)} images packed. Class distribution:")
print(labels_df["label"].value_counts())

## 3. Run the benchmark: baseline -> 2 active-learning cycles

Same protocol as the GalaxyMNIST run: `--anomaly_classes 1` selects the extended-source class as the anomaly (matching `label_idx=1` from `build_seed_cutouts.py`'s `CLASS_NAMES = {0: "point", 1: "extended"}`).

In [ ]:
os.environ["ANOMALYMATCH_PRED_BATCH_SIZE"] = "200"
os.environ["ANOMALYMATCH_NUM_WORKERS"] = "0"  # avoids a DataLoader fork hang seen on Kaggle - see GalaxyMNIST notebook's debugging history

In [ ]:
%cd paper_scripts
!python paper_benchmark.py \
  --dataset chandra --anomaly_classes 1 \
  --n_samples 20 --anomaly_ratio 0.2 \
  --train_iterations 10 --training_runs 2 --n_mislabeled 5 \
  --size 224 --skip_mock_ui --seed 0

## 4. Results

Compare against the GalaxyMNIST run's numbers (baseline AUROC ~0.61, final AUROC ~0.63, top-1% precision ~29% vs. the paper's ~94% at larger scale/more cycles) as the reference point for "does this method do anything useful on this data" - not against the paper's headline number, which used a much larger seed set and more cycles than either of these validation runs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

candidates = sorted(Path("/kaggle/working").rglob("results_summary.csv"))
if not candidates:
    raise FileNotFoundError(
        "No results_summary.csv found under /kaggle/working - scroll the benchmark "
        "cell's output to its end and check it actually finished."
    )
results_csv = candidates[-1]
print("Using:", results_csv)
summary = pd.read_csv(results_csv)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = range(len(summary))

auroc_col = "auroc" if "auroc" in summary.columns else "final_auroc"
auprc_col = "auprc" if "auprc" in summary.columns else "final_auprc"
axes[0].plot(x, summary[auroc_col], marker="o", label="AUROC")
axes[0].plot(x, summary[auprc_col], marker="o", label="AUPRC")
axes[0].set_xlabel("evaluation point (0 = baseline)")
axes[0].set_ylabel("score")
axes[0].set_title("AUROC / AUPRC across AL cycles")
axes[0].legend()

top1_col = [c for c in summary.columns if "top_1" in c and "precision" in c]
if top1_col:
    axes[1].plot(x, summary[top1_col[0]], marker="o", color="tab:red")
    axes[1].axhline(29.3, color="gray", linestyle="--", label="GalaxyMNIST validation run (29.3%)")
    axes[1].set_xlabel("evaluation point (0 = baseline)")
    axes[1].set_ylabel("top-1% precision (%)")
    axes[1].set_title("Top-1% anomaly precision")
    axes[1].legend()

plt.tight_layout()
plt.savefig("chandra_validation_summary.png", dpi=150)
plt.show()

## Next steps

If AUROC/AUPRC/top-1% precision improve across cycles here - and especially if they're comparable to or better than the GalaxyMNIST validation run - that's real evidence AnomalyMatch's FixMatch+active-learning approach transfers to X-ray imaging, the first time this method has been applied to Chandra data. Caveats to keep in mind when interpreting the result (see the intro markdown): `extent_flag` is a real but proxy label, not a vetted anomaly ground truth, and the images carry a known, accepted ACIS streak artifact. Any genuinely interesting candidate (high-scoring extended source the model surfaces) should be cross-checked against SIMBAD/NED before any novelty claim, per `image_anomaly_detection/PLAN.md`'s kill condition and setup checklist.